In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

CATALOG = "smart_claims_sesion_5"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

spark = SparkSession.builder.appName("smart_claims").getOrCreate()
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

# customers
customers_silver = (
    spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers")
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("date_of_birth", F.try_to_date(F.col("date_of_birth"), "dd-MM-yyyy"))
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("lastname", F.trim(F.split(F.col("name"), ",")[0]))
    .withColumn("firstname", F.trim(F.split(F.col("name"), ",")[1]))
    .withColumn("address", F.concat_ws(" ", F.col("borough"), F.col("zip_code")))
    .withColumn("zip_code", F.col("zip_code").cast("string"))
)

customers_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.customers_clean")
print("customers_clean listo")

# policies
policies_silver = (
    spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.policies")
    .withColumn("POLICY_NO", F.col("POLICY_NO").cast("string"))
    .withColumn("CUST_ID", F.col("CUST_ID").cast("int"))
    .withColumn("MODEL_YEAR", F.expr("try_cast(MODEL_YEAR as int)"))
    .withColumn("SUM_INSURED", F.col("SUM_INSURED").cast("double"))
    .withColumn("PREMIUM", F.col("PREMIUM").cast("double"))
    .withColumn("DEDUCTABLE", F.col("DEDUCTABLE").cast("double"))
    .withColumn("POL_ISSUE_DATE", F.to_date(F.col("POL_ISSUE_DATE")))
    .withColumn("POL_EFF_DATE", F.to_date(F.col("POL_EFF_DATE")))
    .withColumn("POL_EXPIRY_DATE", F.to_date(F.col("POL_EXPIRY_DATE")))
    .withColumn("CHASSIS_NO", F.col("CHASSIS_NO").cast("string"))
)

policies_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.policies_clean")
print("policies_clean listo")

# claims
claims_silver = (
    spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.claims")
    .withColumn("claim_no", F.col("claim_no").cast("string"))
    .withColumn("policy_no", F.col("policy_no").cast("string"))
    .withColumn("claim_date", F.to_date(F.col("claim_date")))
    .withColumn("license_issue_date", F.to_date(F.col("license_issue_date")))
    .withColumn("date", F.to_date(F.col("date")))
    .withColumn("age", F.col("age").cast("int"))
    .withColumn("injury", F.col("injury").cast("double"))
    .withColumn("property", F.col("property").cast("double"))
    .withColumn("vehicle", F.col("vehicle").cast("double"))
    .withColumn("total", F.col("total").cast("double"))
    .withColumn("number_of_vehicles_involved", F.col("number_of_vehicles_involved").cast("int"))
    .withColumn("number_of_witnesses", F.col("number_of_witnesses").cast("int"))
    .withColumn("suspicious_activity", F.col("suspicious_activity").cast("boolean"))
    .withColumn("months_as_customer", F.col("months_as_customer").cast("int"))
    .withColumn("hour", F.col("hour").cast("int"))
)

claims_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.claims_clean")
print("claims_clean listo")

# telematics
telematics_silver = (
    spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.telematics")
    .withColumn("event_timestamp", F.col("event_timestamp").cast("timestamp"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("speed", F.col("speed").cast("double"))
    .filter(F.col("latitude").between(-90, 90))
    .filter(F.col("longitude").between(-180, 180))
)

telematics_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.telematics_clean")
print("telematics_clean listo")

# training images
training_silver = (
    spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.training_images")
    .withColumn("image_name", F.regexp_extract(F.col("path"), r"([^/]+)$", 1))
    .withColumn("label", F.regexp_extract(F.col("image_name"), r"^\d+_([^.]+)", 1))
)

training_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.training_images")
print("training_images listo")

# claim images
claim_images_silver = (
    spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images")
    .withColumn("image_name", F.regexp_extract(F.col("path"), r"([^/]+)$", 1))
)

claim_images_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.claim_images")
print("claim_images listo")

# image metadata
image_metadata_silver = (
    spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.image_metadata")
    .withColumn("image_name", F.trim(F.col("image_name")))
    .withColumn("image_id", F.col("image_id").cast("int"))
    .withColumn("claim_no", F.trim(F.col("claim_no")))
    .withColumn("chassis_no", F.trim(F.col("chassis_no")))
    .select("image_name", "image_id", "claim_no", "chassis_no")
)

image_metadata_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.claim_images_metadata_clean")
print("claim_images_metadata_clean listo")

print("Silver lista")

In [0]:
# Validación de datos Silver

# 1. Conteo de registros por tabla
print("=== CONTEO DE REGISTROS ===")
print(f"customers_clean: {spark.table(f'{CATALOG}.{SILVER_SCHEMA}.customers_clean').count():,}")
print(f"policies_clean: {spark.table(f'{CATALOG}.{SILVER_SCHEMA}.policies_clean').count():,}")
print(f"claims_clean: {spark.table(f'{CATALOG}.{SILVER_SCHEMA}.claims_clean').count():,}")
print(f"telematics_clean: {spark.table(f'{CATALOG}.{SILVER_SCHEMA}.telematics_clean').count():,}")
print(f"training_images: {spark.table(f'{CATALOG}.{SILVER_SCHEMA}.training_images').count():,}")
print(f"claim_images: {spark.table(f'{CATALOG}.{SILVER_SCHEMA}.claim_images').count():,}")
print(f"claim_images_metadata_clean: {spark.table(f'{CATALOG}.{SILVER_SCHEMA}.claim_images_metadata_clean').count():,}")

print("\n=== MUESTRA DE DATOS ===")

# 2. Customers
print("\n--- Customers (primeros 5) ---")
spark.table(f"{CATALOG}.{SILVER_SCHEMA}.customers_clean").limit(5).display()

# 3. Policies
print("\n--- Policies (primeros 5) ---")
spark.table(f"{CATALOG}.{SILVER_SCHEMA}.policies_clean").limit(5).display()

# 4. Claims
print("\n--- Claims (primeros 5) ---")
spark.table(f"{CATALOG}.{SILVER_SCHEMA}.claims_clean").limit(5).display()

# 5. Verificar valores NULL en columnas críticas
print("\n=== VERIFICACIÓN DE CALIDAD ===")

print("\n--- NULL counts en Customers ---")
customers_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.customers_clean")
for col_name in customers_df.columns:
    null_count = customers_df.filter(F.col(col_name).isNull()).count()
    print(f"{col_name}: {null_count:,} nulls")

print("\n--- NULL counts en Policies ---")
policies_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.policies_clean")
for col_name in policies_df.columns:
    null_count = policies_df.filter(F.col(col_name).isNull()).count()
    print(f"{col_name}: {null_count:,} nulls")

In [0]:
from pyspark.sql import functions as F

CATALOG = "smart_claims_sesion_5"
SILVER_SCHEMA = "silver"

# fechas y campos derivados en customers
print("customers - fechas y campos derivados")
spark.table(f"{CATALOG}.{SILVER_SCHEMA}.customers_clean") \
    .select("customer_id", "date_of_birth", "name", "firstname", "lastname", "address") \
    .limit(5).show(truncate=False)

# fechas en claims
print("claims - fechas")
spark.table(f"{CATALOG}.{SILVER_SCHEMA}.claims_clean") \
    .select("claim_no", "claim_date", "license_issue_date", "date") \
    .limit(5).show(truncate=False)

# timestamps y coordenadas en telematics
print("telematics - timestamps y coordenadas")
spark.table(f"{CATALOG}.{SILVER_SCHEMA}.telematics_clean") \
    .select("chassis_no", "event_timestamp", "latitude", "longitude", "speed") \
    .limit(5).show(truncate=False)

# label en training images
print("training_images - label")
spark.table(f"{CATALOG}.{SILVER_SCHEMA}.training_images") \
    .select("path", "image_name", "label") \
    .limit(5).show(truncate=False)

# image_name en claim images
print("claim_images - image_name")
spark.table(f"{CATALOG}.{SILVER_SCHEMA}.claim_images") \
    .select("path", "image_name") \
    .limit(5).show(truncate=False)

# metadata lista para unir
print("claim_images_metadata_clean - columnas clave")
spark.table(f"{CATALOG}.{SILVER_SCHEMA}.claim_images_metadata_clean") \
    .select("image_name", "image_id", "claim_no", "chassis_no") \
    .limit(5).show(truncate=False)